In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
    pipeline
)
import evaluate
import torch

In [6]:
#note: I am using CPU has I dont have a GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")

PyTorch: 2.13.0+cpu
CUDA available: False
Using: cpu


In [11]:
#filepath to csv with gold labelled data
GOLD_CSV = "../data/backfill_watched_gold_labelled.csv"
 
df = pd.read_csv(GOLD_CSV)
 
# remove any rows that dont have gold label and ensure label is int
df = df.dropna(subset=["gold_label"])
df["gold_label"] = df["gold_label"].astype(int)
print("total examples" , len(df))
print(f"Label distribution:\n{df['gold_label'].value_counts()}")
df.head()

total examples 433
Label distribution:
gold_label
0    224
1    209
Name: count, dtype: int64


,did,cursor,created_at,text,langs,is_reply,has_embed,matched_keyword,silver_label,gold_label
0,did:plc:nn4m6r3lse2dgznctqaaggcr,25177060456,2026-08-27T17:49:05.995Z,Yesterday I watched a movie called Caroushell ...,['en'],True,False,movie,1,1
1,did:plc:r4jv6ovwzxhfx7xfwelz7dwa,25177063724,2026-08-27T17:49:11.709Z,That movie was way better than I expected it t...,['en'],True,False,movie was,1,1
2,did:plc:dwgtratdmddcphw2iohyjzp2,25177066115,2026-08-27T17:49:13.445Z,The latest Insidious movie gave me everything ...,['en'],False,True,movie,0,1
3,did:plc:t4rzjqmgg2kh4ascstykedoz,25177072210,2026-08-27T17:49:31.923Z,Forever baffled that WB wanted to scrap this m...,['en'],False,True,movie,1,1
4,did:plc:5jcsvspxakd2zcjp7rudzpuf,25177075542,2026-08-27T17:49:37.330Z,Okay so the reason for the redubbing as appare...,['en'],True,False,movie,1,1


In [14]:
# train/validation split (80/20, stratified, seeded for reproducibility)
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=40, 
    stratify=df["gold_label"]
)
print(f"train: {len(train_df)} (positive: {train_df['gold_label'].sum()}, negative: {(train_df['gold_label']==0).sum()})")
print(f"validation:   {len(val_df)} (positive: {val_df['gold_label'].sum()}, negative: {(val_df['gold_label']==0).sum()})")

train: 346 (positive: 167, negative: 179)
validation:   87 (positive: 42, negative: 45)


In [15]:
#convert the df to a Huggingface dataset
train_ds = Dataset.from_pandas(train_df[["text", "gold_label"]].rename(
    columns={"gold_label": "label"}
), preserve_index=False)

val_ds = Dataset.from_pandas(val_df[["text", "gold_label"]].rename(
    columns={"gold_label": "label"}
), preserve_index=False)

In [21]:
#found model Twitter-roBERTa-base which seems suitable for this use-case: https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment
model_roberta = "cardiffnlp/twitter-roberta-base"
model_distilbert = "distilbert-base-uncased"

#documentation: https://huggingface.co/docs/transformers/en/fast_tokenizers

tokenizer = AutoTokenizer.from_pretrained(model_roberta)

#determine the max tokens per post in dataset to use as max_length variabe in tokenisation
#all posts get padded to the max_length value, so need to balance keeping this value as low as possible
#to optimise performance

all_texts = train_df["text"].tolist() + val_df["text"].tolist()
lengths = [len(tokenizer.encode(t)) for t in all_texts]
max_length_in_ds = max(lengths)
print(f"Max tokens from posts in datasets is {max_length_in_ds}")
 
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length_in_ds,
    )
 
train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)

Max tokens from posts in datasets is 124


Map: 100%|██████████| 87/87 [00:00<00:00, 2455.98 examples/s]


In [41]:
#load the model (remember to reload before re-running)
model = AutoModelForSequenceClassification.from_pretrained(
    model_roberta,
    num_labels=2,
    id2label={0: "not_relevant", 1: "viewing_reaction"},
    label2id={"not_relevant": 0, "viewing_reaction": 1},
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6349.54it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized becau

In [43]:
MODEL_OUTPUT_DIR = "../models/watched-classifier"

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

#function which gets called by the trainer at end of each epoch (which passes models prediction and true labels)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    #converts logits into predictions using argmax
    preds = np.argmax(logits, axis=-1)
    #return dictionary of performance metrics
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"],
        "precision": precision_metric.compute(predictions=preds, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels)["recall"],
    }


#Transformers Trainer documentation: https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments


training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    optim="adamw_torch", #this is default optimiser - AdamW is the standard optimizer for transformer fine-tuning
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5, #this is a standard learning rate for fine-tuning a pre-trained model
    weight_decay=0.05, # first run was 0.01, #increase to prevent overfitting. Typical values: 0.01 (standard), 0.1 (stronger regularization), 0.0 (no regularization)
    warmup_steps=0.1,  # 10% of training steps used for linear warmup from 0 to learning_rate. Helps to protect the pre-trained models weights
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1", #using f1 as this is a balanced metric
    greater_is_better=True,
    logging_steps=10,
    seed=40,
    save_total_limit=2,  # keep only the 2 most recent checkpoints to avoid filling disk
    fp16=False,  #default is False but setting this explicitly - set it to True if using GPU
    dataloader_num_workers=0, #num parallel workers, might cause issues on my Windows CPU
)
 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],#rem to adjust this when adjusting num epochs
)

In [44]:
#perform model training

print("*.*.* Starting training *.*.*")
train_result = trainer.train()
 
print(f"\n*.*.* !Training complete! *.*.*")
print(f"Training loss: {train_result.training_loss:.4f}")

*.*.* Starting training *.*.*


c:\Users\adamd\miniconda3\envs\movie-sentiments\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.656826,0.653393,0.517241,0.000000,0.000000,0.000000
2,0.468402,0.402981,0.816092,0.800000,0.842105,0.761905
3,0.296600,0.920633,0.724138,0.760000,0.655172,0.904762
4,0.110302,0.919505,0.781609,0.795699,0.725490,0.880952


c:\Users\adamd\miniconda3\envs\movie-sentiments\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]
c:\Users\adamd\miniconda3\envs\movie-sentiments\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.11s/it]
c:\Users\adamd\miniconda3\envs\movie-sentiments\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|


*.*.* !Training complete! *.*.*
Training loss: 0.3893


In [45]:
#evaluate model on validation set

eval_results = trainer.evaluate()
print(f"\nValidation results:")
for key, value in eval_results.items():
    if key.startswith("eval_"):
        print(f"  {key}: {value:.4f}")

c:\Users\adamd\miniconda3\envs\movie-sentiments\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.110302,0.402981,4,0.816092,0.800000,0.842105,0.761905



Validation results:
  eval_loss: 0.4030
  eval_accuracy: 0.8161
  eval_f1: 0.8000
  eval_precision: 0.8421
  eval_recall: 0.7619


In [46]:
# run the model over the validation set and produce classification_report and confusion matrix

preds_output = trainer.predict(val_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids
 
print("\classification report:")
print(classification_report(
    labels, preds,
    target_names=["not_relevant", "viewing_reaction"]
))
 
print("Confusion Matrix:")
print(confusion_matrix(labels, preds))

<>:7: SyntaxWarning: invalid escape sequence '\c'
<>:7: SyntaxWarning: invalid escape sequence '\c'
C:\Users\adamd\AppData\Local\Temp\ipykernel_16304\550874139.py:7: SyntaxWarning: invalid escape sequence '\c'
  print("\classification report:")
c:\Users\adamd\miniconda3\envs\movie-sentiments\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


\classification report:
                  precision    recall  f1-score   support

    not_relevant       0.80      0.87      0.83        45
viewing_reaction       0.84      0.76      0.80        42

        accuracy                           0.82        87
       macro avg       0.82      0.81      0.81        87
    weighted avg       0.82      0.82      0.82        87

Confusion Matrix:
[[39  6]
 [10 32]]


In [47]:
SAVE_PATH = "../models/viewing-reaction-classifier-final"
#auto save best model based on earlier params set - check model training outputs to see which epoch is chosen
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\nModel saved to {SAVE_PATH}")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.86s/it]



Model saved to ../models/viewing-reaction-classifier-final


In [49]:
#run the model on new text
 
modev_v1 = "../models/viewing-reaction-classifier-final_v1"
modev_v2 = "../models/viewing-reaction-classifier-final_v2"

classifier = pipeline(
    "text-classification",
    model=modev_v1,
    tokenizer=modev_v1,
    device=-1,  # CPU
)
 
test_posts = [
    "I watched Severence yesterday and it was mid",
    "Cant wait to watch spider man tomorrow",
    "We finally watched Barbie lastg night and it was great!!!",
    "Anyone have movie recommendations for tonight?",
    "That film was incredible, no notes, 10/10",
    "Like my photo!? Shot on 35mm film",
    "Make your favourite film sound as boring as possible",
    "Finally watched sopranos",
    "watching my girlfriend eat soup out of a mug, wtf?"
]
 
print("\nTest predictions:")
for post in test_posts:
    result = classifier(post)[0]
    label = result["label"]
    score = result["score"]
    print(f"  [{label} ({score:.2f})] {post}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 699.54it/s]



Test predictions:
  [viewing_reaction (0.98)] I watched Severence yesterday and it was mid
  [viewing_reaction (0.91)] Cant wait to watch spider man tomorrow
  [viewing_reaction (0.98)] We finally watched Barbie lastg night and it was great!!!
  [not_relevant (0.72)] Anyone have movie recommendations for tonight?
  [viewing_reaction (0.98)] That film was incredible, no notes, 10/10
  [not_relevant (0.86)] Like my photo!? Shot on 35mm film
  [not_relevant (0.99)] Make your favourite film sound as boring as possible
  [viewing_reaction (0.95)] Finally watched sopranos
  [not_relevant (0.86)] watching my girlfriend eat soup out of a mug, wtf?
